# 02 — IC / ICIR Analysis

**Purpose**: Visualise the feature information coefficient (IC) and IC Information Ratio (ICIR) across all three return horizons, and examine factor decay.

Prerequisite: run `scripts/factor_research.py` first to generate `reports/factor_research_{1y,3y,5y}.csv`.

In [ ]:
import sys, warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings('ignore')
ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

horizons = ['1y', '3y', '5y']
reports = {h: ROOT / 'reports' / f'factor_research_{h}.csv' for h in horizons}
dfs = {}
for h, path in reports.items():
    if path.exists():
        dfs[h] = pd.read_csv(path)
        print(f'{h}: {len(dfs[h])} features loaded from {path.name}')
    else:
        print(f'MISSING: {path} — run scripts/factor_research.py')

## 1 — Top features by |ICIR| (all horizons)

In [ ]:
for h, df in dfs.items():
    if 'icir' not in df.columns:
        continue
    top = df.sort_values('icir', key=abs, ascending=False).head(20)
    print(f'\n--- {h} horizon: top 20 by |ICIR| ---')
    print(top[['feature', 'mean_ic', 'std_ic', 'icir', 'n_years', 'pct_positive_ic']].to_string(index=False))

## 2 — ICIR bar chart (top 30 per horizon)

In [ ]:
fig, axes = plt.subplots(1, len(dfs), figsize=(7 * len(dfs), 10))
if len(dfs) == 1:
    axes = [axes]

for ax, (h, df) in zip(axes, dfs.items()):
    if 'icir' not in df.columns:
        continue
    top = df.sort_values('icir', key=abs, ascending=False).head(30)
    colors = ['steelblue' if v > 0 else 'crimson' for v in top['icir']]
    ax.barh(top['feature'][::-1], top['icir'][::-1], color=colors[::-1])
    ax.axvline(0, color='black', linewidth=0.8)
    ax.axvline(0.5, color='green', linestyle='--', linewidth=0.8, label='ICIR=0.5')
    ax.axvline(-0.5, color='green', linestyle='--', linewidth=0.8)
    ax.set_title(f'{h} — Top 30 by |ICIR|')
    ax.set_xlabel('ICIR')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 3 — IC stability (pct_positive_ic) vs mean_ic scatter

In [ ]:
fig, axes = plt.subplots(1, len(dfs), figsize=(7 * len(dfs), 6))
if len(dfs) == 1:
    axes = [axes]

for ax, (h, df) in zip(axes, dfs.items()):
    if 'mean_ic' not in df.columns or 'pct_positive_ic' not in df.columns:
        continue
    ax.scatter(df['mean_ic'], df['pct_positive_ic'], alpha=0.5, s=20, color='steelblue')
    ax.axhline(0.60, linestyle='--', color='orange', linewidth=0.8, label='60% stable')
    ax.axvline(0.02, linestyle='--', color='green', linewidth=0.8, label='IC>=0.02')
    ax.axvline(-0.02, linestyle='--', color='green', linewidth=0.8)
    ax.set_title(f'{h} — IC stability vs mean IC')
    ax.set_xlabel('mean_ic')
    ax.set_ylabel('pct_positive_ic')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 4 — Feature overlap across horizons

In [ ]:
import json

feat_sets = {}
for h in horizons:
    p = ROOT / 'models' / f'feature_sets_{h}.json'
    if p.exists():
        with open(p) as f:
            feat_sets[h] = set(json.load(f)['features'])
        print(f'{h}: {len(feat_sets[h])} selected features')

if len(feat_sets) == 3:
    shared_all = feat_sets['1y'] & feat_sets['3y'] & feat_sets['5y']
    print(f'\nFeatures in ALL three horizons: {len(shared_all)}')
    for f in sorted(shared_all):
        print(f'  {f}')

## 5 — Feature selection summary (IC / ICIR / PSI)

In [ ]:
summary_path = ROOT / 'reports' / 'feature_selection_summary.csv'
if summary_path.exists():
    sel = pd.read_csv(summary_path)
    print(f'feature_selection_summary.csv: {len(sel)} rows')
    print(sel.groupby(['horizon', 'selected']).size().unstack(fill_value=0))
    
    for h, grp in sel.groupby('horizon'):
        selected = grp[grp['selected'] == True]
        print(f'\n{h} selected features (sorted by |ICIR|):')
        top = selected.sort_values('icir', key=abs, ascending=False)
        print(top[['feature', 'mean_ic', 'icir', 'psi_train_vs_test']].head(20).to_string(index=False))